In [14]:
from pathlib import Path
import os
import shutil
import cv2
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from skimage.feature import hog

In [2]:
images = []

In [3]:
for path in Path("data/extracted_squares/extracted_pieces").rglob("*.jpg"):
    images.append(path)

In [4]:
len(images)

3520

In [5]:
def normalize(image_path):
    img = cv2.imread(str(image_path))
    patch = cv2.resize(img, (128, 128))
    patch = cv2.cvtColor(patch, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(patch)
    l = cv2.equalizeHist(l)
    return cv2.merge([l, a, b])

In [7]:
def extract_hog(patch):
    gray  = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    feat  = hog(gray, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), visualize=False)
    return feat

In [8]:
features = []

In [9]:
for img_path in images:
    norm_img = normalize(img_path)
    feat = extract_hog(norm_img)
    features.append(feat)

In [10]:
features = np.array(features)

In [11]:
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

In [12]:
pca = PCA(n_components=50)
features_pca = pca.fit_transform(features_scaled)

In [16]:
kmeans = KMeans(n_clusters=14, n_init=20)
labels = kmeans.fit_predict(features_pca)

In [17]:
for path, label in zip(images, labels):
    dest = f"data/clusters/{label}/"
    os.makedirs(dest, exist_ok=True)
    shutil.copy(path, dest)